# Подготовка трекера - Edge-путь (ByteTrack)

Путь для RK3588: детектор RKNN (NPU) + **ByteTrack** (`supervision`).
ByteTrack без ReID и без torch - работает на edge без GPU.

In [ ]:
import itertools
import json
from pathlib import Path
import cv2
import numpy as np

if not hasattr(np, "asfarray"):
    np.asfarray = lambda a, dtype=float: np.asarray(a, dtype=dtype)

import pandas as pd
import motmetrics as mm
import supervision as sv
from ultralytics import YOLO
from tqdm.notebook import tqdm

ROOT = Path().resolve().parent
MOT20_DIR = ROOT / "data" / "MOT20" / "MOT20" / "train"
MODELS_DIR = ROOT / "models"
MODEL_WEIGHTS = MODELS_DIR / "yolo11n_mot20_v2.pt"

DET_CONF, DET_IOU, DET_IMGSZ, DEVICE, FRAME_RATE = 0.25, 0.45, 640, "cuda", 25
EVAL_SEQUENCES = ["MOT20-01", "MOT20-02", "MOT20-03", "MOT20-05"]
MAX_FRAMES_GRID = 300
TARGET_IDF1 = 0.75

assert MODEL_WEIGHTS.exists(), f"Нет весов {MODEL_WEIGHTS} — запусти detector_training.ipynb"
MODEL_WEIGHTS

In [ ]:
def load_mot_gt(gt_path):
    """GT MOT20 -> {frame_id: [(track_id, x, y, w, h), ...]}"""
    gt = {}
    for line in gt_path.read_text().strip().splitlines():
        p = line.split(",")
        if int(p[6]) == 0 or int(p[7]) != 1 or (len(p) > 8 and float(p[8]) < 0.25):
            continue
        gt.setdefault(int(p[0]), []).append((int(p[1]), float(p[2]), float(p[3]), float(p[4]), float(p[5])))
    return gt


def run_bytetrack(model, seq_dir, tracker, max_frames=None):
    """Детектор + ByteTrack по кадрам -> {frame_id: [(track_id, x, y, w, h), ...]}"""
    frames = sorted((seq_dir / "img1").glob("*.jpg"))
    if max_frames:
        frames = frames[:max_frames]
    preds = {}
    for ff in frames:
        r = model.predict(source=cv2.imread(str(ff)), imgsz=DET_IMGSZ, conf=DET_CONF, iou=DET_IOU, device=DEVICE, verbose=False, classes=[0])[0]
        if r.boxes is not None and len(r.boxes):
            dets = sv.Detections(xyxy=r.boxes.xyxy.cpu().numpy(), confidence=r.boxes.conf.cpu().numpy(), class_id=np.zeros(len(r.boxes), dtype=int))
            tracked = tracker.update_with_detections(dets)
        else:
            tracker.update_with_detections(sv.Detections.empty())
            tracked = sv.Detections.empty()
        fp = []
        if tracked.tracker_id is not None:
            for i in range(len(tracked.xyxy)):
                x1, y1, x2, y2 = tracked.xyxy[i]
                fp.append((int(tracked.tracker_id[i]), float(x1), float(y1), float(x2 - x1), float(y2 - y1)))
        preds[int(ff.stem)] = fp
    return preds


def mot_metrics(gt, pred):
    """IDF1 / MOTA / IDSW / Precision / Recall через motmetrics"""
    acc = mm.MOTAccumulator(auto_id=True)
    for fid in sorted(set(gt) | set(pred)):
        g, h = gt.get(fid, []), pred.get(fid, [])
        gb = np.array([[o[1], o[2], o[3], o[4]] for o in g]) if g else np.empty((0, 4))
        hb = np.array([[o[1], o[2], o[3], o[4]] for o in h]) if h else np.empty((0, 4))
        acc.update([o[0] for o in g], [o[0] for o in h], mm.distances.iou_matrix(gb, hb, max_iou=0.5))
    s = mm.metrics.create().compute(
        acc, metrics=["idf1", "mota", "motp", "num_switches", "precision", "recall"], name="e")
    return {"idf1": float(s["idf1"].iloc[0]), "mota": float(s["mota"].iloc[0]), "idsw": int(s["num_switches"].iloc[0]), "prec": float(s["precision"].iloc[0]), "rec": float(s["recall"].iloc[0])}


model = YOLO(str(MODEL_WEIGHTS)).to(DEVICE)

## Grid search параметров ByteTrack на MOT20-01

In [ ]:
GRID = {
    "track_activation_threshold": [0.20, 0.25, 0.30],
    "minimum_matching_threshold": [0.7, 0.8, 0.85, 0.9],
    "lost_track_buffer": [20, 30, 50, 75],
}
seq01 = MOT20_DIR / "MOT20-01"
gt01 = load_mot_gt(seq01 / "gt" / "gt.txt")
gt01_grid = {k: v for k, v in gt01.items() if k <= MAX_FRAMES_GRID}

rows = []
for combo in tqdm(list(itertools.product(*GRID.values())), desc="grid"):
    kw = dict(zip(GRID.keys(), combo))
    pred = run_bytetrack(model, seq01, sv.ByteTrack(**kw, frame_rate=FRAME_RATE), MAX_FRAMES_GRID)
    rows.append({**kw, **mot_metrics(gt01_grid, pred)})

df_grid = pd.DataFrame(rows).sort_values("idf1", ascending=False).reset_index(drop=True)
df_grid.head(10)

## Валидация лучших параметров на 4 последовательностях MOT20

In [ ]:
best = df_grid.iloc[0]
BEST = {"track_activation_threshold": float(best["track_activation_threshold"]),
        "minimum_matching_threshold": float(best["minimum_matching_threshold"]),
        "lost_track_buffer": int(best["lost_track_buffer"])}

rows = []
for seq in tqdm(EVAL_SEQUENCES, desc="sequences"):
    s = MOT20_DIR / seq
    pred = run_bytetrack(model, s, sv.ByteTrack(**BEST, frame_rate=FRAME_RATE))
    rows.append({"sequence": seq, **mot_metrics(load_mot_gt(s / "gt" / "gt.txt"), pred)})

df_final = pd.DataFrame(rows)
mean_idf1 = df_final["idf1"].mean()
df_final

In [ ]:
params = {**BEST, "frame_rate": FRAME_RATE, "det_conf": DET_CONF, "det_iou": DET_IOU,
          "model_weights": MODEL_WEIGHTS.name,
          "idf1_mot20_train": round(float(mean_idf1), 4),
          "mota_mot20_train": round(float(df_final["mota"].mean()), 4),
          "target_idf1_met": bool(mean_idf1 >= TARGET_IDF1)}
out = MODELS_DIR / "bytetrack_best_params.json"
out.write_text(json.dumps(params, indent=2, ensure_ascii=False))
params